# vLLM Quickstart

## Installation

In [ ]:
!pip show vllm

## Basic Usage

In [ ]:
from vllm import LLM

llm = LLM(
    model="facebook/opt-1.3b",   # use a small model
    trust_remote_code=True
)

outputs = llm.generate("Hello from M3 Mac!", max_tokens=32)
print(outputs[0].outputs[0].text)

## Setting Up Paths

In [ ]:
import sys
import os
# Get directory path where vllm is installed
# You can find this with `which vllm` in terminal
vllm_path = "/Users/vinotganesan/vllm_env/bin/vllm"  # Replace with actual path from `which vllm`
sys.path.append(os.path.dirname(vllm_path))
# Now imports should work
from vllm import LLM, SamplingParams

## Troubleshooting: Using Transformers as Alternative

When vLLM installation has C++ extension compatibility issues, you can use Transformers as an alternative:

In [ ]:
# WORKAROUND: The vLLM installation has C++ extension compatibility issues
# The V1 engine requires custom operations that aren't properly compiled
# Solution: Rebuild vLLM from source with compatible PyTorch
#
# To fix permanently, run in terminal:
# source ~/vllm_env/bin/activate
# cd ~/vllm
# pip uninstall vllm -y
# pip install -e . --no-build-isolation
#
# For now, using transformers as alternative for testing:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading model with transformers (alternative to vLLM)...")

# Load model and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="./models")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir="./models",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

# Generate text
prompt = "Write a short poem about artificial intelligence."
inputs = tokenizer(prompt, return_tensors="pt")

print("\nGenerating...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.95,
        do_sample=True
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nGenerated text:")
print(generated_text[len(prompt):])

## Notes

In [ ]:
- When working with vLLM, you may encounter C++ extension compatibility issues
- If you experience issues, consider rebuilding from source with:
  source ~/vllm_env/bin/activate
  cd ~/vllm
  pip uninstall vllm -y
  pip install -e . --no-build-isolation

- As an alternative, you can use the HuggingFace Transformers library directly

- For M3 Mac compatibility, use models that support CPU inference or ensure proper PyTorch/CUDA setup

## Quick Model Loading Examples

In [ ]:
from vllm import LLM, SamplingParams

# Small model for testing
llm = LLM(
    model="facebook/opt-125m",
    trust_remote_code=True
)

# Configure sampling
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=100,
    top_p=0.95
)

# Generate
prompts = ["What is machine learning?", "Explain neural networks"]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(output.outputs[0].text)

## Key Takeaways

- vLLM provides fast inference with optimized batching

- Use appropriate model sizes for your hardware

- Configure sampling parameters for desired output quality

- Monitor memory usage and adjust accordingly

- For production use, consider deploying via API server